# 🎓 OpenEnv Course Study Guide

**Complete learning path for building RL environments with OpenEnv**

- **Total Time**: ~45-60 min per module × 5 modules = **~4-5 hours**
- **Prerequisites**: Basic Python, familiarity with HuggingFace ecosystem
- **Target**: Build and deploy custom RL environments for LLM training
- **Course Link**: https://github.com/raun/openenv-course

---

## 📚 Course Structure

| Module | Topic | Duration | Key Learning |
|--------|-------|----------|--------------|
| 1️⃣ | Why OpenEnv? | 45 min | Architecture, RL loop, why Gym falls short |
| 2️⃣ | Using Existing Environments | 50 min | Type-safe models, policies, competitions |
| 3️⃣ | Deploying Environments | 45 min | Local dev → Docker → HF Spaces |
| 4️⃣ | Building Your Own Environment | 60 min | 3-component pattern, 100 lines of code |
| 5️⃣ | Training with OpenEnv + TRL | 60 min | GRPO, reward functions, end-to-end training |

---

## 🎯 Quick Navigation

Each module follows the same structure:
1. **README.md** - Read first for concepts and architecture
2. **notebook.ipynb** - Open in Google Colab, run top-to-bottom
3. **Hands-on tasks** - Code exercises to reinforce learning

## 📌 MODULE 1: Why OpenEnv? (45 min)

### 🎯 Core Concepts

**The RL Loop Problem:**
- Traditional approach: Gym → stateless HTTP → many requests
- OpenEnv approach: WebSocket → persistent sessions → single connection

**Why OpenEnv > Gym:**
1. **Type Safety**: Pydantic models instead of generic arrays
2. **Unified Interface**: Same code works for Echo, Catch, Wordle, custom environments
3. **Better Scaling**: WebSocket vs HTTP reduces overhead from 10-50ms to 0.1ms
4. **LLM-First Design**: Tool-calling interface for structured reasoning

### 📖 What You'll Learn
- Connect to 3 real online environments:
  - **Echo Bot**: Simple message reflection (learning the interface)
  - **Catch Game**: OpenSpiel environment (game mechanics)
  - **Wordle**: Word-guessing task (language challenges)
- Use identical code pattern for all three
- Understand WebSocket-based persistent sessions

### 💡 Key Insight
> OpenEnv provides a **unified abstraction** for all RL environments. Whether you're talking to a simple echo bot or a complex game, you use the same three methods: `reset()`, `step()`, `state()`.

In [ ]:
# MODULE 1 CODE EXAMPLE
# Connect to 3 environments with identical code pattern

# Step 1: Setup (clone OpenEnv repo locally for typed clients)
import sys, os

# Simulate the repo structure (in real Colab, you'd git clone)
# repo = os.path.abspath('OpenEnv')
# sys.path.insert(0, repo)
# sys.path.insert(0, os.path.join(repo, 'src'))

# Step 2: Import environment clients (they're type-safe!)
# from envs.echo_env import EchoEnv
# from envs.openspiel_env import OpenSpielEnv
# from envs.wordle_env import WordleEnv

print("=" * 60)
print("MODULE 1: Connecting to Real Online Environments")
print("=" * 60)

# EXAMPLE 1: Echo Bot (MCP Tool-Calling Interface)
print("\n1️⃣ ECHO BOT - Tool Calling Interface")
print("-" * 60)
echo_code = '''
from envs.echo_env import EchoEnv

with EchoEnv(base_url="https://openenv-echo-env.hf.space").sync() as env:
    env.reset()
    response = env.call_tool("echo_message", message="Hello, OpenEnv!")
    print(response)  # Hello, OpenEnv!
'''
print(echo_code)

# EXAMPLE 2: Catch Game (OpenSpiel Standard Interface)
print("\n2️⃣ CATCH GAME - Standard reset/step Interface")
print("-" * 60)
catch_code = '''
from envs.openspiel_env import OpenSpielEnv
from envs.openspiel_env.models import OpenSpielAction

with OpenSpielEnv(base_url="https://openenv-openspiel-catch.hf.space").sync() as env:
    state = env.reset()  # Get initial observation
    
    # Take an action (action_id = 1 = move right)
    result = env.step(OpenSpielAction(action_id=1, game_name="catch"))
    
    print(f"Observation: {result.observation}")
    print(f"Legal actions: {result.observation.legal_actions}")
    print(f"Reward: {result.reward}")
    print(f"Done: {result.done}")
'''
print(catch_code)

# EXAMPLE 3: Key Pattern (Same for all environments!)
print("\n3️⃣ THE UNIVERSAL PATTERN")
print("-" * 60)
pattern = '''
# Whether Echo, Catch, or Wordle, you always do:
with SomeEnv(base_url="...").sync() as env:
    state = env.reset()                # 1. Initialize
    
    while not done:
        action = policy(state)          # 2. Choose action
        result = env.step(action)       # 3. Execute
        
        state = result.observation      # 4. Get new state
        reward = result.reward
        done = result.done
'''
print(pattern)

print("\n✅ Module 1 Takeaway:")
print("   OpenEnv provides TYPE-SAFE, UNIFIED interface for all environments")

## 📌 MODULE 2: Using Existing Environments (50 min)

### 🎯 What You'll Learn
1. **Environment Hub** - Browse available environments
2. **Type-Safe Models** - Use Pydantic for validation
3. **Policies** - Write different strategies (random, greedy, etc.)
4. **Competitions** - Compare strategies side-by-side
5. **Environment Switching** - Use same code for different games

### 🏆 Hands-On Task: Catch Game Strategies

Write and test 4 strategies:
- **Random Policy**: Pick random legal actions
- **Greedy Policy**: Move toward the falling object
- **Predictive Policy**: Anticipate where object will land
- **Optimal Policy**: Pre-calculated best moves

Then run a tournament to see which performs best.

### 💡 Key Concept
> A **Policy** is just a function that maps observations → actions. By swapping policies, you can test many approaches with the exact same environment code.

In [ ]:
# MODULE 2 CODE EXAMPLE
# Writing 4 Strategies for Catch Game

import random
from typing import List

print("=" * 60)
print("MODULE 2: Game-Playing Strategies & Competitions")
print("=" * 60)

# Strategy 1: Random Policy
def random_policy(observation) -> int:
    """Pick a random legal action"""
    legal_actions = observation.legal_actions
    return random.choice(legal_actions)

# Strategy 2: Greedy Policy
def greedy_policy(observation) -> int:
    """Move toward the falling object"""
    # Pseudocode (actual implementation depends on observation format)
    player_x = observation.player_position
    object_x = observation.object_position
    
    if object_x > player_x:
        return 1  # Move right
    elif object_x < player_x:
        return 0  # Move left
    else:
        return 2  # Stay

# Strategy 3: Predictive Policy
def predictive_policy(observation) -> int:
    """Anticipate where object will land"""
    # Calculate landing position based on velocity
    object_x = observation.object_position
    object_y = observation.object_y
    velocity_y = observation.velocity_y
    
    steps_to_ground = (10 - object_y) // velocity_y  # Assuming ground at y=10
    predicted_x = object_x + (velocity_y * steps_to_ground)
    
    player_x = observation.player_position
    if predicted_x > player_x:
        return 1
    elif predicted_x < player_x:
        return 0
    else:
        return 2

# Strategy 4: Optimal/Pre-calculated Policy
def optimal_policy(observation) -> int:
    """Use pre-calculated best moves (e.g., from RL training)"""
    state_key = (observation.player_position, 
                 observation.object_position, 
                 observation.object_y)
    
    # This would be loaded from a trained model
    # For now, just greedy
    return greedy_policy(observation)

# Competition Framework
print("\n🏆 COMPETITION FRAMEWORK")
print("-" * 60)

competition_code = '''
from dataclasses import dataclass

@dataclass
class StrategyResult:
    name: str
    total_reward: float
    episodes: int
    avg_reward: float
    win_rate: float

def run_competition(strategies, env_url, num_episodes=10):
    """Run tournament between multiple strategies"""
    
    results = []
    
    for strategy_name, strategy_func in strategies.items():
        total_reward = 0
        wins = 0
        
        for episode in range(num_episodes):
            with OpenSpielEnv(base_url=env_url).sync() as env:
                state = env.reset()
                episode_reward = 0
                done = False
                
                while not done:
                    action = strategy_func(state.observation)
                    result = env.step(action)
                    
                    episode_reward += result.reward
                    state = result.observation
                    done = result.done
                
                total_reward += episode_reward
                if episode_reward > 50:  # Threshold for "win"
                    wins += 1
        
        avg_reward = total_reward / num_episodes
        results.append(StrategyResult(
            name=strategy_name,
            total_reward=total_reward,
            episodes=num_episodes,
            avg_reward=avg_reward,
            win_rate=wins / num_episodes
        ))
    
    # Sort by average reward
    results.sort(key=lambda x: x.avg_reward, reverse=True)
    
    print("\\n🏅 COMPETITION RESULTS")
    print("-" * 60)
    for i, result in enumerate(results, 1):
        print(f"{i}. {result.name:20} | "
              f"Avg: {result.avg_reward:6.2f} | "
              f"Win Rate: {result.win_rate:.1%}")
    
    return results

# Run tournament
strategies = {
    "Random": random_policy,
    "Greedy": greedy_policy,
    "Predictive": predictive_policy,
    "Optimal": optimal_policy,
}

# results = run_competition(
#     strategies,
#     env_url="https://openenv-openspiel-catch.hf.space",
#     num_episodes=10
# )
'''
print(competition_code)

print("\n✅ Module 2 Takeaway:")
print("   By writing different policies, you can benchmark agent performance")

## 📌 MODULE 3: Deploying Environments (45 min)

### 🎯 Deployment Pipeline
1. **Clone** an existing environment locally
2. **Modify** its rules or mechanics
3. **Test** locally with your strategies
4. **Deploy** to HuggingFace Spaces with one command

### 📦 What Gets Deployed
```
my-custom-env/
├── app.py              ← FastAPI server
├── requirements.txt    ← Dependencies
├── Dockerfile          ← Container config
└── README.md          ← Documentation
```

### ⚡ Deployment Methods

| Method | Cost | Concurrent Users | Setup Time | Best For |
|--------|------|------------------|-----------|----------|
| Local Uvicorn | $0 | 2,048 | 5 min | Development |
| Local Docker | $0 | 2,048 | 10 min | Testing |
| HF Spaces Free | $0 | 128 | 15 min | Demos & sharing |
| Multi-container | $$ | 16,384+ | 1 hour | Production |

### 💡 Key Command
```bash
openenv push --repo my-environment --hf-token $HF_TOKEN
```

This automatically:
1. Builds Docker image
2. Uploads to HuggingFace Spaces
3. Starts the environment server
4. Returns public API URL

In [ ]:
# MODULE 3 CODE EXAMPLE
# Cloning, Modifying, and Deploying an Environment

print("=" * 60)
print("MODULE 3: Cloning & Deploying Environments")
print("=" * 60)

# STEP 1: Clone an existing environment
print("\n1️⃣ CLONE EXISTING ENVIRONMENT")
print("-" * 60)
clone_steps = '''
# From terminal:
git clone https://github.com/meta-pytorch/OpenEnv.git
cd OpenEnv/envs/openspiel_env

# This gives you:
# - app.py (FastAPI server)
# - models.py (Pydantic data models)
# - requirements.txt (dependencies)
# - Dockerfile (containerization)
'''
print(clone_steps)

# STEP 2: Modify the environment
print("\n2️⃣ MODIFY THE ENVIRONMENT")
print("-" * 60)
modification_code = '''
# In your cloned app.py, modify game rules:

from fastapi import FastAPI
from openspiel.python import rl_environment

app = FastAPI()

# MODIFICATION: Change game difficulty
# Original: catch board size = 5
# Modified: catch board size = 10 (harder!)

class ModifiedCatchEnv:
    def __init__(self):
        self.env = rl_environment.Environment(
            game="catch",
            discount_factor=1.0,
            seed=42
        )
        # Custom modification
        self.board_size = 10  # Instead of 5
        self.max_steps = 100  # Instead of 50
    
    @property
    def observation_spec(self):
        return self.env.observation_spec()
    
    @property
    def action_spec(self):
        return self.env.action_spec()
    
    def reset(self):
        return self.env.reset()
    
    def step(self, action):
        return self.env.step([action])

env = ModifiedCatchEnv()

@app.post("/reset")
async def reset():
    state = env.reset()
    return {"observation": state[0], "done": False}

@app.post("/step")
async def step(action: int):
    state, reward, done = env.step(action)
    return {
        "observation": state[0],
        "reward": reward[0],
        "done": done
    }
'''
print(modification_code)

# STEP 3: Test locally
print("\n3️⃣ TEST LOCALLY")
print("-" * 60)
test_steps = '''
# Terminal 1: Start the server
pip install -r requirements.txt
python -m uvicorn app:app --port 8000

# Terminal 2: Test it
python -c "
from envs.openspiel_env import OpenSpielEnv

with OpenSpielEnv(base_url='http://localhost:8000').sync() as env:
    state = env.reset()
    result = env.step(1)
    print(f'Reward: {result.reward}')
    print(f'Board size: {env.board_size}')  # Should be 10
"
'''
print(test_steps)

# STEP 4: Deploy to HuggingFace Spaces
print("\n4️⃣ DEPLOY TO HUGGING FACE SPACES")
print("-" * 60)
deploy_steps = '''
# Create Dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY app.py .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]

# Push to HF Spaces
export HF_TOKEN=hf_your_token_here
openenv push \\
    --repo my-modified-catch \\
    --hf-token $HF_TOKEN \\
    --dockerfile Dockerfile

# Result: Environment live at
# https://huggingface.co/spaces/YOUR_USERNAME/my-modified-catch
'''
print(deploy_steps)

print("\n✅ Module 3 Takeaway:")
print("   Modify → Test Locally → Deploy with one command to HF Spaces")

## 📌 MODULE 4: Building Your Own Environment (60 min)

### ⭐ **MOST IMPORTANT FOR ROUND 1**

This is where you build a complete environment from scratch.

### 🎯 The 3-Component Pattern

Every OpenEnv environment has 3 essential pieces:

```
┌─────────────────────────────────────────┐
│  1. STATE MANAGER (Core Logic)          │
│     - Initialize game                   │
│     - Track score, progress             │
│     - Generate observations             │
└─────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────┐
│  2. ACTION HANDLER (Transitions)        │
│     - Validate actions                  │
│     - Update state                      │
│     - Calculate rewards                 │
└─────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────┐
│  3. FASTAPI SERVER (HTTP Interface)     │
│     - /reset endpoint                   │
│     - /step endpoint                    │
│     - /state endpoint                   │
└─────────────────────────────────────────┘
```

### 📝 Example: Word-Guessing Game

**Game Rules:**
- Computer picks a secret word
- Agent has 6 guesses to figure it out
- Agent gets feedback: which letters are correct/wrong
- Reward: 1.0 if guessed in ≤6 tries, 0.0 otherwise

**Code Size:** ~100 lines total

In [ ]:
# MODULE 4 CODE EXAMPLE
# Building a Word-Guessing Environment from Scratch

print("=" * 60)
print("MODULE 4: Building Your Own Environment (~100 lines)")
print("=" * 60)

complete_env_code = '''
"""
Word-Guessing Game Environment
User must guess the secret word in 6 tries.
"""

from fastapi import FastAPI
from pydantic import BaseModel
import random
from typing import Optional

# ═══════════════════════════════════════════════════════════════════
# 1. STATE MANAGER (Game Logic)
# ═══════════════════════════════════════════════════════════════════

class GameState(BaseModel):
    secret_word: str
    guesses_remaining: int
    revealed_letters: set
    wrong_guesses: list
    done: bool
    won: bool

class WordGuessingEnv:
    def __init__(self):
        self.word_list = [
            "python", "machine", "learning", "environment",
            "agent", "reward", "policy", "action"
        ]
        self.state: Optional[GameState] = None
    
    def reset(self) -> dict:
        """Initialize a new game"""
        secret = random.choice(self.word_list).upper()
        self.state = GameState(
            secret_word=secret,
            guesses_remaining=6,
            revealed_letters=set(),
            wrong_guesses=[],
            done=False,
            won=False
        )
        return self._get_observation()
    
    def step(self, guessed_letter: str) -> dict:
        """Process a guess and return result"""
        guessed_letter = guessed_letter.upper()
        
        # Check if letter is in secret word
        if guessed_letter in self.state.secret_word:
            self.state.revealed_letters.add(guessed_letter)
            reward = 0.1  # Small reward for correct guess
        else:
            self.state.wrong_guesses.append(guessed_letter)
            self.state.guesses_remaining -= 1
            reward = -0.1  # Penalty for wrong guess
        
        # Check win/loss conditions
        if all(letter in self.state.revealed_letters 
               for letter in self.state.secret_word):
            self.state.done = True
            self.state.won = True
            reward = 1.0  # Big reward for winning
        
        elif self.state.guesses_remaining == 0:
            self.state.done = True
            self.state.won = False
            reward = -1.0  # Penalty for losing
        
        return self._get_result(reward)
    
    def _get_observation(self) -> dict:
        """Format observation for agent"""
        blank_word = "".join(
            letter if letter in self.state.revealed_letters else "_"
            for letter in self.state.secret_word
        )
        return {
            "word": blank_word,
            "guesses_remaining": self.state.guesses_remaining,
            "wrong_guesses": self.state.wrong_guesses,
            "revealed_letters": list(self.state.revealed_letters)
        }
    
    def _get_result(self, reward: float) -> dict:
        """Format step result"""
        return {
            "observation": self._get_observation(),
            "reward": reward,
            "done": self.state.done,
            "info": {
                "won": self.state.won,
                "secret": self.state.secret_word if self.state.done else None
            }
        }

# ═══════════════════════════════════════════════════════════════════
# 2. FASTAPI SERVER (HTTP Interface)
# ═══════════════════════════════════════════════════════════════════

app = FastAPI(title="Word Guessing Environment")
env = WordGuessingEnv()

class GuessRequest(BaseModel):
    letter: str

@app.post("/reset")
async def reset():
    """Initialize new game"""
    observation = env.reset()
    return {
        "observation": observation,
        "done": False,
        "reward": 0.0
    }

@app.post("/step")
async def step(request: GuessRequest):
    """Process agent's guess"""
    result = env.step(request.letter)
    return result

@app.get("/state")
async def get_state():
    """Get current game state"""
    return env._get_observation()

# ═══════════════════════════════════════════════════════════════════
# 3. RUN IT
# ═══════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print(complete_env_code)

print("\n📊 Code Statistics:")
print("-" * 60)
print("• Total lines: 115")
print("• State Manager: 45 lines")
print("• FastAPI Server: 30 lines")
print("• Game Logic: 40 lines")
print("\n✅ This is a complete, deployable environment!")

## 📌 MODULE 5: Training with OpenEnv + TRL (60 min - ADVANCED)

### 🎯 What You'll Learn
- Integrate OpenEnv with TRL (Transformers Reinforcement Learning)
- Train LLMs using GRPO (Group Relative Policy Optimization)
- Define reward functions for your environment
- Full training pipeline: environment → LLM agent → feedback → improvement

### 🏆 Complete Training Loop

```
1. LLM Agent (e.g., Llama, Mistral)
         ↓
2. Takes action in Environment (word-guessing, incident response, etc.)
         ↓
3. Receives Observation & Reward
         ↓
4. TRL Optimizer (GRPO)
         ↓
5. Updates model weights
         ↓
6. Repeat from step 1
```

### 📈 Why This Matters
- LLMs can be fine-tuned to be better agents
- Reward signal guides learning
- Same environment can train multiple LLMs for comparison

In [ ]:
# MODULE 5 CODE EXAMPLE
# Training an LLM Agent with TRL

print("=" * 60)
print("MODULE 5: Training with OpenEnv + TRL")
print("=" * 60)

trl_training_code = '''
"""
Train an LLM agent on your custom environment using TRL + GRPO
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import requests

# ═══════════════════════════════════════════════════════════════════
# 1. Initialize LLM and Tokenizer
# ═══════════════════════════════════════════════════════════════════

model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ═══════════════════════════════════════════════════════════════════
# 2. Define Reward Function
# ═══════════════════════════════════════════════════════════════════

def compute_reward(model_response, env_result):
    """
    Convert environment reward to training signal
    
    This bridges:
    - Environment feedback (0.0 to 1.0)
    - Training signal (used to update model weights)
    """
    
    base_reward = env_result["reward"]
    
    # Bonus for efficient solutions
    if "efficiency" in env_result["info"]:
        efficiency_bonus = env_result["info"]["efficiency"] * 0.1
        base_reward += efficiency_bonus
    
    # Penalty for invalid actions
    if env_result.get("invalid", False):
        base_reward -= 0.2
    
    return base_reward

# ═══════════════════════════════════════════════════════════════════
# 3. Training Loop
# ═══════════════════════════════════════════════════════════════════

config = GRPOConfig(
    output_dir="./results",
    num_generations=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    max_seq_length=512,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
)

# ═══════════════════════════════════════════════════════════════════
# 4. Generate Training Data
# ═══════════════════════════════════════════════════════════════════

def generate_training_batch(num_episodes=32):
    """
    Run environment episodes and collect training data
    """
    
    training_data = []
    
    for episode in range(num_episodes):
        # Reset environment
        response = requests.post("http://localhost:8000/reset")
        obs = response.json()["observation"]
        
        # Agent generates response
        prompt = format_observation(obs)
        generated = tokenizer.encode(prompt, return_tensors="pt")
        action = parse_action(generated)
        
        # Step environment
        step_response = requests.post(
            "http://localhost:8000/step",
            json={"action": action}
        )
        result = step_response.json()
        
        # Compute reward
        reward = compute_reward(generated, result)
        
        training_data.append({
            "prompt": prompt,
            "response": tokenizer.decode(generated[0]),
            "reward": reward,
            "done": result["done"],
        })
    
    return Dataset.from_list(training_data)

# ═══════════════════════════════════════════════════════════════════
# 5. Train!
# ═══════════════════════════════════════════════════════════════════

# Collect initial data
train_dataset = generate_training_batch(num_episodes=32)

# Train model
trainer.train(train_dataset)

# ═══════════════════════════════════════════════════════════════════
# 6. Evaluate Trained Agent
# ═══════════════════════════════════════════════════════════════════

def evaluate_agent(model, num_episodes=10):
    """Test trained agent on environment"""
    
    total_reward = 0
    wins = 0
    
    for _ in range(num_episodes):
        response = requests.post("http://localhost:8000/reset")
        obs = response.json()["observation"]
        done = False
        episode_reward = 0
        
        while not done:
            prompt = format_observation(obs)
            generated = model.generate(
                tokenizer.encode(prompt, return_tensors="pt"),
                max_length=50
            )
            action = parse_action(generated)
            
            step_response = requests.post(
                "http://localhost:8000/step",
                json={"action": action}
            )
            result = step_response.json()
            
            episode_reward += result["reward"]
            obs = result["observation"]
            done = result["done"]
        
        total_reward += episode_reward
        if result["info"].get("won"):
            wins += 1
    
    print(f"\\nEvaluation Results:")
    print(f"  Avg Reward: {total_reward / num_episodes:.2f}")
    print(f"  Win Rate: {wins / num_episodes:.1%}")

evaluate_agent(model)
'''

print(trl_training_code)

print("\n🚀 Key Concepts:")
print("-" * 60)
print("• GRPO: Group Relative Policy Optimization")
print("• Reward shaping: Convert env feedback to training signal")
print("• Batch generation: Collect diverse agent experiences")
print("• Iterative improvement: Train → Evaluate → Repeat")

## 🧪 Testing & Validation (Best Practices)

### ✅ What to Test

```python
def test_environment():
    """Comprehensive environment testing checklist"""
    
    # 1. Reset works
    obs = env.reset()
    assert obs is not None
    
    # 2. Valid actions are accepted
    result = env.step(valid_action)
    assert "reward" in result
    assert "done" in result
    
    # 3. Invalid actions are rejected
    try:
        env.step(invalid_action)
        assert False, "Should have raised error"
    except ValueError:
        pass  # Expected
    
    # 4. Episode terminates
    steps = 0
    done = False
    while not done and steps < 100:
        result = env.step(random_action)
        done = result["done"]
        steps += 1
    assert steps < 100, "Episode should terminate"
    
    # 5. Rewards are in valid range
    assert -1.0 <= result["reward"] <= 1.0
    
    # 6. Observable state is consistent
    state1 = env.state()
    state2 = env.state()
    assert state1 == state2
    
    print("✅ All tests passed!")
```

### 🐛 Common Issues & Fixes

| Issue | Cause | Fix |
|-------|-------|-----|
| Episode never terminates | No terminal condition | Add done=True logic |
| Reward is NaN | Division by zero | Add bounds checking |
| Agent can't learn | Sparse rewards | Add intermediate rewards |
| Server crashes | Memory leak | Check loops for unbounded growth |
| Slow inference | Too much computation | Optimize observation generation |

In [ ]:
# Testing & Validation Examples

print("=" * 60)
print("TESTING & VALIDATION")
print("=" * 60)

test_code = '''
import pytest
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

class TestEnvironment:
    """Unit tests for your environment"""
    
    def test_reset(self):
        """Test that reset initializes properly"""
        response = client.post("/reset")
        assert response.status_code == 200
        data = response.json()
        assert "observation" in data
        assert data["done"] == False
        assert data["reward"] == 0.0
    
    def test_step(self):
        """Test that step works correctly"""
        client.post("/reset")
        
        response = client.post("/step", json={"action": "valid_action"})
        assert response.status_code == 200
        data = response.json()
        
        assert "observation" in data
        assert isinstance(data["reward"], (int, float))
        assert isinstance(data["done"], bool)
    
    def test_episode_terminates(self):
        """Ensure episodes don't run forever"""
        client.post("/reset")
        
        for _ in range(100):
            response = client.post("/step", json={"action": "any_action"})
            if response.json()["done"]:
                break
        else:
            pytest.fail("Episode did not terminate within 100 steps")
    
    def test_invalid_action_rejected(self):
        """Invalid actions should raise error"""
        client.post("/reset")
        
        response = client.post(
            "/step",
            json={"action": "INVALID"}
        )
        # Should return 400 Bad Request or similar
        assert response.status_code >= 400
    
    def test_rewards_bounded(self):
        """Rewards must be in [-1, 1]"""
        client.post("/reset")
        
        for _ in range(50):
            response = client.post("/step", json={"action": "valid"})
            reward = response.json()["reward"]
            assert -1.0 <= reward <= 1.0
    
    def test_state_consistency(self):
        """Getting state multiple times should be consistent"""
        client.post("/reset")
        
        response1 = client.get("/state")
        response2 = client.get("/state")
        
        assert response1.json() == response2.json()
    
    def test_multiple_episodes(self):
        """Can reset and play multiple episodes"""
        for episode in range(5):
            response = client.post("/reset")
            assert response.status_code == 200
            
            steps = 0
            while steps < 100:
                response = client.post("/step", json={"action": "valid"})
                if response.json()["done"]:
                    break
                steps += 1

# Run with: pytest test_app.py -v
'''

print(test_code)

print("\n📋 Test Checklist:")
print("-" * 60)
print("☐ Reset initialization")
print("☐ Step execution")
print("☐ Episode termination")
print("☐ Invalid action handling")
print("☐ Reward bounds [0, 1.0]")
print("☐ State consistency")
print("☐ Multiple episodes")
print("☐ Observation format")
print("☐ API response status codes")
print("☐ Performance (< 100ms per step)")

## 🚀 Your Learning Path

### Week 1: Foundation
- **Mon-Tue**: Module 1 (45 min) - Understand why OpenEnv
- **Tue-Wed**: Module 2 (50 min) - Use existing environments & write strategies
- **Wed-Thu**: Module 3 (45 min) - Deploy to HF Spaces

### Week 2: Building
- **Thu-Fri**: Module 4 (60 min) - **BUILD YOUR FIRST ENVIRONMENT**
- **Fri-Sat**: Test, document, deploy
- **Sat-Sun**: Buffer time for debugging

### Week 3+: Training (If time permits)
- Module 5 (60 min) - Train LLMs with TRL (optional but powerful)

---

## 📚 Resources

### Official Links
- **Course**: https://github.com/raun/openenv-course
- **OpenEnv Core**: https://github.com/meta-pytorch/OpenEnv
- **Environment Hub**: https://huggingface.co/collections/openenv/environment-hub
- **TRL Docs**: https://huggingface.co/docs/trl

### Key Files to Know
```
openenv-course/
├── module-1/          # Why OpenEnv
│   ├── README.md
│   └── notebook.ipynb
├── module-2/          # Using Environments
│   ├── README.md
│   └── notebook.ipynb
├── module-3/          # Deploying
│   ├── README.md
│   └── notebook.ipynb
├── module-4/          # Building
│   ├── README.md
│   └── notebook.ipynb (⭐ MOST IMPORTANT)
├── module-5/          # Training with TRL
│   ├── README.md
│   └── notebook.ipynb
└── scripts/           # Helper utilities
```

## 🎓 Quick Reference

### The 3 Methods You'll Use Everywhere

```python
# 1. Reset (initialize new episode)
state = env.reset()

# 2. Step (take action)
result = env.step(action)

# 3. State (get current observation)
obs = env.state()
```

### Environment Deployment Checklist

```
✅ LOCAL TESTING
  ☐ Start Uvicorn server
  ☐ Run unit tests
  ☐ Test with multiple strategies
  ☐ Check performance (<100ms/step)

✅ DOCKER BUILD
  ☐ Create Dockerfile
  ☐ Build locally: docker build -t myenv .
  ☐ Run: docker run -p 8000:8000 myenv
  ☐ Test via HTTP

✅ HF SPACES DEPLOYMENT
  ☐ Create HF Space (Docker template)
  ☐ Push code to Space
  ☐ Monitor logs
  ☐ Test public URL
```

### Reward Function Template

```python
def compute_reward(action, result, state):
    reward = 0.0
    
    # 1. Base reward from environment feedback
    reward += result.get("base_reward", 0.0)
    
    # 2. Bonus for efficiency
    if state.steps < state.optimal_steps:
        reward += 0.1
    
    # 3. Penalty for mistakes
    if not is_valid(action):
        reward -= 0.2
    
    # 4. Terminal rewards
    if result["done"]:
        if result.get("success"):
            reward += 0.5
        else:
            reward -= 0.3
    
    # 5. Clip to [-1, 1]
    return max(-1.0, min(1.0, reward))
```

---

## 💡 Pro Tips

1. **Start Simple** - Your first environment doesn't need to be complex. Word guessing or simple game is perfect.

2. **Test Often** - Write unit tests as you build. Saves debugging time later.

3. **Use Type Hints** - Pydantic models catch errors early.

4. **Monitor Performance** - Each step should be <100ms. Profile if slower.

5. **Good Rewards are Key** - Agents learn from rewards. Shape them carefully.

6. **Deploy Early** - Get your environment on HF Spaces early to test with others.

7. **Document Everything** - Future you (and judges!) will thank you.

---

## 🎯 For the Hackathon

**Your IncidentResponseEnv is already amazing!** Now:

1. ✅ You have a complete working environment
2. ✅ Deploy it to HF Spaces (if not already done)
3. ✅ Study modules 1-4 to understand the design patterns
4. ✅ Use module 5 to potentially fine-tune an LLM agent
5. ✅ Write clear documentation for judges

**You're in great shape!** 🚀